In [1]:
#visualise differences between the Raw and Training-Ready versions of ERA5

import matplotlib.pyplot as plt
import numpy as np
from anemoi.datasets import open_dataset
import time
import torch
from einops import rearrange

In [2]:
ds_data = open_dataset('https://data.ecmwf.int/anemoi-datasets/era5-o96-1979-2023-6h-v8.zarr')
ds_data

zarr https://data.ecmwf.int/anemoi-datasets/era5-o96-1979-2023-6h-v8.zarr


In [3]:
#attributes of this Anemoi dataset
print(f"Resolution:    {ds_data.resolution}")
print(f"Frequency:     {ds_data.frequency}")
print(f"Grid points:   {sum(ds_data.grids):,}")
print(f"Time steps:    {len(ds_data.dates):,}  ({ds_data.dates[0]} to {ds_data.dates[-1]})")
print(f"Variables:     {len(ds_data.variables)}")
print(f"Shape:         (dates={len(ds_data.dates)}, variables={len(ds_data.variables)}, ensemble=1, gridpoints={sum(ds_data.grids)})")

Resolution:    O96
Frequency:     6:00:00
Grid points:   40,320
Time steps:    65,744  (1979-01-01T00:00:00 to 2023-12-31T18:00:00)
Variables:     101
Shape:         (dates=65744, variables=101, ensemble=1, gridpoints=40320)


In [4]:
#simulate a training sample: a window of consecutive time steps
rollout_steps = 5  #e.g. 1 input + 4 forecast steps
start_idx = np.random.randint(0, len(ds_data.dates) - rollout_steps)
#or can also use: anemoi-training's DataReader.get_sample()

#load a temporal window (what Dataloader fetches per sample)
x = ds_data[start_idx : start_idx + rollout_steps]
print(f"Window shape:   {x.shape}  — (dates, variables, ensemble, gridpoints)")

#(dates, variables, ensemble, gridpoints) -> (dates, ensemble, gridpoints, variables)
x = rearrange(x, 'dates variables ensemble gridpoints -> dates ensemble gridpoints variables')
x = torch.from_numpy(x)

print(f"Model input shape:  {x.shape}  — (dates, ensemble, gridpoints, variables)")
print(f"Dtype:              {x.dtype}")
print(f"Memory per sample:  {x.nbytes / 1024**2:.1f} MB")

Window shape:   (5, 101, 1, 40320)  — (dates, variables, ensemble, gridpoints)
Model input shape:  torch.Size([5, 1, 40320, 101])  — (dates, ensemble, gridpoints, variables)
Dtype:              torch.float32
Memory per sample:  77.7 MB


In [5]:
#apply normalisation using built-in statistics
stats = ds_data.statistics
data_mean = torch.from_numpy(stats['mean']).float()
data_std = torch.from_numpy(stats['stdev']).float()

#normalise
x_norm = (x - data_mean) / data_std

In [6]:
#dataset before and after normalisation for z500
print(f"Before normalisation — mean: {x[0, 0, :, 96].mean():.2f}, std: {x[0, 0, :, 96].std():.2f}  (variable: {ds_data.variables[96]})")
print(f"After normalisation  — mean: {x_norm[0, 0, :, 96].mean():.2f}, std: {x_norm[0, 0, :, 96].std():.2f}")

Before normalisation — mean: 55382.87, std: 2822.35  (variable: z_500)
After normalisation  — mean: -0.10, std: 1.04
